In [18]:
#!/usr/bin/env python
# coding: utf-8
import Arm_Lib
import cv2 as cv
import threading
from time import sleep
import ipywidgets as widgets
from IPython.display import display
from color_sorting import color_sorting
import json
import pickle
from dofbot_config import *


In [19]:
import Arm_Lib
Arm = Arm_Lib.Arm_Device()
joints_0 = [90, 135, 0, 10, 90, 0]
Arm.Arm_serial_servo_write6_array(joints_0, 1000)

In [20]:
sorting = color_sorting()
model = 'General'

In [21]:
HSV_path="HSV_config.txt"

# color_hsv  = {"red"   : ((0, 143, 145), (13, 255, 255)),
#               "green" : ((60, 143, 145), (68, 255, 255)),
#               "blue"  : ((100, 143, 145), (119, 255, 255)),
#               "yellow": ((26, 143, 145), (28, 255, 255))}
# with open('HSV_Config.pkl', 'wb') as f:
#     pickle.dump(color_hsv, f)

with open('HSV_Config.pkl', 'rb') as f:
    color_hsv = pickle.load(f)

try: read_HSV(HSV_path,color_hsv)
except Exception as e: print("Read HSV_config Error!!!", e)


In [22]:
model = "General"
button_layout = widgets.Layout(width='320px', height='60px', align_self='center')
output = widgets.Output()

exit_button = widgets.Button(description='Exit', button_style='danger', layout=button_layout)
imgbox = widgets.Image(format='jpg', height=480, width=640, layout=widgets.Layout(align_self='center'))
controls_box = widgets.VBox([imgbox, exit_button], layout=widgets.Layout(align_self='center'))

In [23]:
def exit_button_Callback(value):
    global model
    model = 'Exit'
    with output: print(model)
exit_button.on_click(exit_button_Callback)

In [24]:
import numpy as np
def change_brightness(img, alpha, beta):
   return cv.addWeighted(img, alpha, np.zeros(img.shape, img.dtype),0, beta)

def camera():
    # Open camera
    capture = cv.VideoCapture(0)
    while capture.isOpened():
        try:
            _, img = capture.read()
            img = cv.resize(img, (640, 480))
            img = cv.flip(img, 1)
            show_img = img.copy()
            #r=(147,320,239,160)
            #img= img[int(r[1]):int(r[1]+r[3]),int(r[0]):int(r[0]+r[2])]
            img = change_brightness(img, 1, 0)
            if sorting.status == 'waiting':
                sorting.Sorting_grap(img, color_hsv)
                #sorting.sorting_run(color_hsv)
            if model == 'Exit':
                cv.destroyAllWindows()
                capture.release()
                break
            imgbox.value = cv.imencode('.jpg', show_img)[1].tobytes()
        except KeyboardInterrupt:capture.release()

In [25]:
display(controls_box,output)
threading.Thread(target=camera, ).start()

Output()

[ WARN:3@315.144] global /io/opencv/modules/videoio/src/cap_v4l.cpp (902) open VIDEOIO(V4L2:/dev/video0): can't open camera by index


In [9]:




##### 

In [26]:
#!/usr/bin/env python
# coding: utf-8

import cv2 as cv
import numpy as np
import threading
import time
import os
import pickle
import ipywidgets as widgets
from IPython.display import display

import Arm_Lib
from color_sorting import color_sorting
from dofbot_config import *

# 1. Initialize Arm & Move to Inspection Stance
Arm = Arm_Lib.Arm_Device()
time.sleep(0.5)
joints_inspect = [90, 135, 0, 10, 90, 30]  # Camera inspection stance
Arm.Arm_serial_servo_write6_array(joints_inspect, 1000)
time.sleep(1.2)

# 2. Initialize Color Sorting Class
sorting = color_sorting()
model = 'General'

# 3. Load or Default HSV Values
HSV_path = "HSV_config.txt"
color_hsv = {
    "red":    ((0, 140, 100), (10, 255, 255)),
    "green":  ((35, 100, 100), (75, 255, 255)),
    "blue":   ((100, 100, 100), (130, 255, 255)),
    "yellow": ((20, 100, 100), (34, 255, 255))
}

# Try loading calibrated HSV file if present; fall back safely if missing
if os.path.exists('HSV_Config.pkl'):
    try:
        with open('HSV_Config.pkl', 'rb') as f:
            color_hsv = pickle.load(f)
    except Exception as e:
        print("[WARN] Could not load HSV_Config.pkl, using defaults:", e)

if os.path.exists(HSV_path):
    try:
        read_HSV(HSV_path, color_hsv)
    except Exception as e:
        print("[WARN] Could not read HSV_config.txt:", e)

# 4. Jupyter UI Widgets
button_layout = widgets.Layout(width='200px', height='45px', align_self='center')
output = widgets.Output()

exit_button = widgets.Button(description='Exit Program', button_style='danger', layout=button_layout)
imgbox = widgets.Image(format='jpeg', width=640, height=480, layout=widgets.Layout(align_self='center'))
controls_box = widgets.VBox([imgbox, exit_button], layout=widgets.Layout(align_self='center'))

def exit_button_callback(b):
    global model
    model = 'Exit'
    with output:
        print("[INFO] Shutting down...")

exit_button.on_click(exit_button_callback)

# 5. Helper Function for Image Processing
def change_brightness(img, alpha=1.0, beta=0):
    return cv.addWeighted(img, alpha, np.zeros(img.shape, img.dtype), 0, beta)

def open_camera():
    """Attempts to find and open the DOFBOT camera securely."""
    for idx in [0, 1]:
        cap = cv.VideoCapture(idx, cv.CAP_V4L2)
        if cap.isOpened():
            ret, _ = cap.read()
            if ret:
                cap.set(cv.CAP_PROP_FRAME_WIDTH, 640)
                cap.set(cv.CAP_PROP_FRAME_HEIGHT, 480)
                return cap
            cap.release()
    return None

# 6. Sorting Action in a Background Worker (Prevents Video Freezing)
def arm_action_thread(img_crop):
    try:
        # Executes Yahboom's grab-and-place routine
        sorting.Sorting_grap(img_crop, color_hsv)
    except Exception as err:
        with output:
            print("[ERROR] Sorting execution error:", err)

# 7. Main Camera Loop
def camera_loop():
    global model, sorting
    
    capture = open_camera()
    if capture is None:
        with output:
            print("[ERROR] Camera could not be opened. Restart your notebook kernel.")
        return

    with output:
        print("[INFO] Camera stream started. System ready.")

    try:
        while capture.isOpened() and model != 'Exit':
            ret, img = capture.read()
            if not ret:
                time.sleep(0.02)
                continue

            img = cv.resize(img, (640, 480))
            img = cv.flip(img, 1)
            processed_img = change_brightness(img, 1.0, 0)
            
            # Yahboom DOFBOT standard inspection ROI crop (X, Y, W, H)
            # Adjust if your camera sits at a different height:
            roi_x, roi_y, roi_w, roi_h = 140, 220, 360, 240
            cv.rectangle(img, (roi_x, roi_y), (roi_x + roi_w, roi_y + roi_h), (0, 255, 0), 2)
            
            # Non-blocking trigger: only run arm motion when idle
            if sorting.status == 'waiting':
                crop_area = processed_img[roi_y:roi_y + roi_h, roi_x:roi_x + roi_w]
                # Start arm motion on a separate thread to keep camera running
                t = threading.Thread(target=arm_action_thread, args=(crop_area,))
                t.daemon = True
                t.start()

            # Push JPEG frames to Jupyter widget
            ret_encode, buf = cv.imencode('.jpg', img, [cv.IMWRITE_JPEG_QUALITY, 80])
            if ret_encode:
                imgbox.value = buf.tobytes()

            time.sleep(0.01)

    except KeyboardInterrupt:
        pass
    finally:
        capture.release()
        cv.destroyAllWindows()
        # Reset arm to rest stance
        joints_rest = [90, 90, 90, 90, 90, 30]
        Arm.Arm_serial_servo_write6_array(joints_rest, 1000)
        with output:
            print("[INFO] Camera closed and DOFBOT safely returned to rest.")

# 8. Start Display and Thread
display(controls_box, output)
threading.Thread(target=camera_loop, daemon=True).start()

Output()

[ WARN:4@353.319] global /io/opencv/modules/videoio/src/cap_v4l.cpp (902) open VIDEOIO(V4L2:/dev/video0): can't open camera by index
